# Visual Exploration of Graph Neural Networks in Your Computational Notebooks

In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import sys, os
sys.path.append(os.path.abspath("src"))

## Visualizing Graphs with Dual Views

- **API**: GraphVisualizer
- **Parameter**: dataFile (the file path of the graph data that you want to visualize). 

In [3]:
from gnn_vis_widgets import GraphVisualizer
w = GraphVisualizer()
w.add_data(dataFile="test_data/test_new_data.json")
w

Loading JSON data from: test_data/test_new_data.json


GraphVisualizer(graphData={'edge_index': [[0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 0, 6, 0, 7, 0, 8, 0, 10, 0, 11, 0, 12…

## Visualizing Large Graphs with Subgraph Dual Views

- **API**:
- GraphVisualizer (class)
- subgraph_hoop_visualizer (API)
- multiple_subgraph_hoop_visualizer (API)
- **Parameter**:
- dataFile (the file path of the graph data that you want to visualize).
- hubNode (the hub node that we want to visualize)
- hubNodes (an array of hub nodes that need to be visualize)
- hoopNum (the hoop number we used for partition)

### Visualizing Large Graphs with Hoop-based Extraction (Single Hub Node)

In [4]:
sub_w = GraphVisualizer()
sub_w.add_data(dataFile="test_data/twitch.json")
sub_w.subgraph_hoop_visualizer(hubNode=0, hoopNum=3)
sub_w

Loading JSON data from: test_data/twitch.json
Updated to 3-hop subgraph centered at 0.


GraphVisualizer(graphData={'x': [[-0.23666860163211823, -0.23071609437465668, -0.16054700314998627, -0.1982001…

### Visualizing Large Graphs with Hoop-based Extraction (Multiple Hub Nodes)

In [5]:
mul_sub_w = GraphVisualizer()
mul_sub_w.add_data(dataFile="test_data/twitch.json")
mul_sub_w.multiple_subgraph_hoop_visualizer(hubNodes=[0, 1], hoopNum=1)
mul_sub_w

Loading JSON data from: test_data/twitch.json
Updated to 1-hop subgraphs centered at [0, 1].


GraphVisualizer(graphData={'x': [[-0.23666860163211823, -0.23071609437465668, -0.16054700314998627, -0.1982001…

## Editing Graph Structures through GraphEditor

- **API**: GraphEditor
- **Parameter**: dataFile (the file path of the graph data that you want to visualize). 

In [6]:
from gnn_vis_widgets import GraphEditor
editor = GraphEditor()
editor.add_data(dataFile="test_data/karate_dataset.json")
editor

Exposing file to browser: /files/test_data/karate_dataset.json


GraphEditor()

### Exporting Data from GraphEditor through DataBridge

In [7]:
import json
full_path = "test_data/karate_dataset.json"
with open(full_path, "r") as f:
    data_before = json.load(f)
len(data_before['x'])

34

In [8]:
editor.export_data_to_json("test_data/test_new_data.json")

Graph data exported to test_data/test_new_data.json


## Visualizing Model Intermediate Features through Matrix View (WIP)

- **API**: GNNVisualizer

### Constructuring a Graph Neural Network using PyTorch Geometric

In [18]:
import os
import torch

from torch_geometric.datasets import KarateClub
dataset = KarateClub()
data = dataset[0]
edge_index = data.edge_index
print(edge_index.t())

from torch.nn import Linear
from torch_geometric.nn import GCNConv


class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        torch.manual_seed(1234)
        self.conv1 = GCNConv(dataset.num_features, 4)
        self.conv2 = GCNConv(4, 4)
        self.conv3 = GCNConv(4, 4)
        self.conv4 = GCNConv(4, 2)
        self.classifier = Linear(2, dataset.num_classes)

    def forward(self, x, edge_index):
        # print("Input shapes:")
        # print("x:", x.shape)
        # print("edge_index:", edge_index.shape)
        outputs = {}
        h = self.conv1(x, edge_index)
        h = h.tanh()
        outputs["conv1"] = h
        h = self.conv2(h, edge_index)
        h = h.tanh()
        outputs["conv2"] = h
        h = self.conv3(h, edge_index)
        h = h.tanh()  # Final GNN embedding space.
        outputs["conv3"] = h
        h = self.conv4(h, edge_index)
        h = h.tanh()
        outputs["conv4"] = h
        # Apply a final (linear) classifier.
        out = self.classifier(h)
        prob = torch.softmax(out, dim=1)
        outputs["final"] = prob
        return outputs

model = GCN()
print("modeling finished")

tensor([[ 0,  1],
        [ 0,  2],
        [ 0,  3],
        [ 0,  4],
        [ 0,  5],
        [ 0,  6],
        [ 0,  7],
        [ 0,  8],
        [ 0, 10],
        [ 0, 11],
        [ 0, 12],
        [ 0, 13],
        [ 0, 17],
        [ 0, 19],
        [ 0, 21],
        [ 0, 31],
        [ 1,  0],
        [ 1,  2],
        [ 1,  3],
        [ 1,  7],
        [ 1, 13],
        [ 1, 17],
        [ 1, 19],
        [ 1, 21],
        [ 1, 30],
        [ 2,  0],
        [ 2,  1],
        [ 2,  3],
        [ 2,  7],
        [ 2,  8],
        [ 2,  9],
        [ 2, 13],
        [ 2, 27],
        [ 2, 28],
        [ 2, 32],
        [ 3,  0],
        [ 3,  1],
        [ 3,  2],
        [ 3,  7],
        [ 3, 12],
        [ 3, 13],
        [ 4,  0],
        [ 4,  6],
        [ 4, 10],
        [ 5,  0],
        [ 5,  6],
        [ 5, 10],
        [ 5, 16],
        [ 6,  0],
        [ 6,  4],
        [ 6,  5],
        [ 6, 16],
        [ 7,  0],
        [ 7,  1],
        [ 7,  2],
        [ 

In [19]:
criterion = torch.nn.CrossEntropyLoss()  # Define loss criterion.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # Define optimizer.

def train(data):
    optimizer.zero_grad()  # Clear gradients.
    outputs = model(data.x, data.edge_index)  # Perform a single forward pass.
    out = outputs["final"]
    h = outputs["conv4"]
    loss = criterion(out[data.train_mask], data.y[data.train_mask])  # Compute the loss solely based on the training nodes.
    loss.backward()  # Derive gradients.
    optimizer.step()  # Update parameters based on gradients.
    return loss, h

for epoch in range(401):
    loss, h = train(data)
print("training is finished")

training is finished


In [20]:
outputs = model(data.x, data.edge_index)

print(f"conv1 shape: {list(outputs['conv1'].shape)}")
print(f"conv2 shape: {list(outputs['conv2'].shape)}")
print(f"conv3 shape: {list(outputs['conv3'].shape)}")
print(f"conv4 shape: {list(outputs['conv4'].shape)}")
print(f"final shape: {list(outputs['final'].shape)}")

conv1 shape: [34, 4]
conv2 shape: [34, 4]
conv3 shape: [34, 4]
conv4 shape: [34, 2]
final shape: [34, 4]


In [21]:
model_info = {
    "gnn_layer_0": data['x'].detach().cpu().numpy().tolist(),
    "gnn_layer_1": outputs['conv1'].detach().cpu().numpy().tolist(),
    "gnn_layer_2": outputs['conv2'].detach().cpu().numpy().tolist(),
    "gnn_layer_3": outputs['conv3'].detach().cpu().numpy().tolist(),
    "gnn_layer_4": outputs['conv4'].detach().cpu().numpy().tolist(),
    "fc_layer_1": outputs['final'].detach().cpu().numpy().tolist(),
}

In [13]:
outputs['final'][0]

tensor([8.1241e-03, 9.7322e-01, 1.9228e-04, 1.8461e-02],
       grad_fn=<SelectBackward0>)

### Visualizing a Graph Neural Network in Computational Notebooks
- **API**: GNNVisualizer
- **Parameters**:
- graphFIle: input graph JSON file.
- weightFile: model weight JSON file.
- modelInfo: the intermediate layer outputs and model architecture information.

**TODO**
- add data processing layer
- add computation symbol to the vis
- clean-up current variable process flow
- [Dec 26]: implement pooling/aggregation FC layers
- [Dec 27]: implement sub-graph computes
- build automation and publish to PyPI

In [25]:
from gnn_vis_widgets import GNNVisualizer

model_w = GNNVisualizer(graphData={}, graphPath="test_data/test_new_data.json", intmData={})
model_w.add_data(
    graphFile="test_data/karate_dataset.json", 
    weightFile="test_data/weights/node_weights.json",
    modelInfo=model_info
)
model_w

Loading JSON data from: test_data/karate_dataset.json
Loading JSON data from: test_data/weights/node_weights.json
graphData: dict_keys(['x', 'edge_index', 'y', 'batch']), intmData: dict_keys(['conv1.bias', 'conv2.bias', 'conv3.bias', 'classifier.weight', 'classifier.bias', 'onnx::MatMul_271', 'onnx::MatMul_274', 'onnx::MatMul_277']) loaded.


GNNVisualizer(graphData={'x': [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0…

### Visualizing Computational Graph for a Single Node
- **API**: NodeComputationalGraph